In [1]:
import os
os.chdir('/content/nbody_exoplanets')
print(os.getcwd())

/content/nbody_exoplanets


In [4]:
!ls output/

final_exoplanets.csv


In [5]:
import os
import multiprocessing

print(f"CPU cores: {multiprocessing.cpu_count()}")
print(f"RAM: {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3):.1f} GB")

CPU cores: 8
RAM: 51.0 GB


In [7]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from numba import jit
import scipy
import pandas as pd

In [8]:
from integrator import Particles
from integrator import Particle
from integrator import constants
from integrator import DataIO
from integrator import Simulation
from integrator import WH_SA_P
from integrator import WH_SB_P
from integrator import WH_SC_P
from integrator import WH_SAB_P
from integrator import WH_SABC_P

In [9]:
from pathlib import Path

In [13]:
dir_out_path = 'output/'

In [16]:
!ls {dir_out_path}

final_exoplanets.csv


In [14]:
Path(dir_out_path,'planets_triples.csv')

PosixPath('output/planets_triples.csv')

In [18]:
import os
full_path = str(Path(dir_out_path, 'final_exoplanets.csv'))
print(repr(full_path))
print(repr(dir_out_path))
print("Exists:", os.path.exists(full_path))


'output/final_exoplanets.csv'
'output/'
Exists: True


In [19]:
planets = pd.read_csv(full_path)

In [20]:
planets.head()

,Unnamed: 0,pl_name,pl_orbper,pl_orbsmax,pl_rade,pl_bmasse,pl_orbeccen,pl_eqt,sy_dist
0,4,GJ 229 A c,121.93268,0.38400,2.8700,8.581367,0.366,NaN,5.75624
1,5,GJ 229 b,579.47495,1.08600,3.9700,14.937935,0.404,NaN,5.75624
2,16,HAT-P-8 b,3.07634,0.04496,15.6926,406.822400,0.000,1713.0,211.55300
3,26,HD 132563 b,1544.00000,2.62000,13.6000,473.547000,0.220,NaN,105.15500
4,38,HD 65216 b,577.60000,1.30100,13.6000,411.589850,0.270,NaN,35.12050


In [22]:
dir_in_path = 'input/'

In [24]:
!ls {dir_in_path}

planets_triple.csv  triple_systems_selected.csv


In [33]:
triples = pd.read_csv('input/triple_systems_selected.csv')

In [34]:
triples.head()

,Sustav,Orbitalna konfiguracija,Oznaka matične zvijezde,Ime zvijezde A,Ime zvijezde B,Ime zvijezde C,Masa A [m_s],Masa B [m_s],Masa C [m_s],Radijus A [R_s],...,i_u [º],Omega_u [º],omega_u [º],Oznake vanjskog para,a_v [AU],e_v,i_v [º],Omega_v [º],omega_v [º],Izvori
0,Gliese 667,S(C),C,Gliese 667 A,Gliese 667 B,Gliese 667 C,0.730,0.690,0.310,0.760,...,127.600,133.200,68.100,AB-C,230.0,NaN,NaN,NaN,NaN,"[14], [17], [18], [19], [20]"
1,LTT 1445,S(A),A,LTT 1445 A,LTT 1445 B,LTT 1445 C,0.257,0.215,0.161,0.271,...,89.640,137.630,209.000,A-BC,34.0,NaN,NaN,NaN,NaN,"[14], [17], [20], [21], [22]"
2,94 Ceti,S(A),A,94 Ceti A,94 Ceti B,94 Ceti C,1.300,0.550,0.340,1.898,...,108.323,191.496,334.895,A-BC,220.0,0.26,104.0,97.0,342.0,"[14], [17], [19], [23], [24], [25]"
3,Kepler-444,S(A),A,Kepler-444 A,Kepler-444 B,Kepler-444 C,0.754,0.307,0.296,0.753,...,NaN,NaN,NaN,A-BC,52.2,0.55,85.4,250.7,227.3,"[14], [17], [19], [20], [26]"
4,HD 132563,S(C),C,HD 132563 Aa,HD 132563 Ab,HD 132563 B,1.081,0.600,1.010,1.300,...,NaN,NaN,160.200,AB-C,400.0,NaN,NaN,NaN,NaN,"[14], [17], [19], [20], [27], [28]"


## Gliese 667

In [48]:
from integrator import run_system

configs = []
for _, row in triples.iterrows():
    pl_subset = planets[planets['pl_name'].str.contains(row['Sustav'])]
    
    # dt = 1/20 of inner binary period (Kepler's 3rd law, solar units -> years)
    a_bin = row['a_u [AU]']
    m_bin = row['Masa A [m_s]'] + row['Masa B [m_s]']
    P_bin = a_bin**1.5 / np.sqrt(m_bin)
    dt = P_bin / 20.0

    configs.append({
        'system_type': row['Orbitalna konfiguracija'],
        'host_star':   row['Oznaka matične zvijezde'],
        'mA': row['Masa A [m_s]'], 'mB': row['Masa B [m_s]'], 'mC': row['Masa C [m_s]'],
        'RA': row['Radijus A [R_s]'], 'RB': row['Radijus B [R_s]'], 'RC': row['Radijus B [R_s]'],
        'TA': row['T_eff A [K]'],    'TB': row['T_eff B [K]'],    'TC': row['T_eff C [K]'],
        'inner_a': row['a_u [AU]'], 'inner_e': row['e_u'],
        'inner_i': row['i_u [º]'], 'inner_Omega': row['Omega_u [º]'],
        'inner_omega': row['omega_u [º]'], 'inner_T0': 0,
        'outer_a': row['a_v [AU]'], 'outer_e': row['e_v'],
        'outer_i': row['i_v [º]'], 'outer_Omega': row['Omega_v [º]'],
        'outer_omega': row['omega_v [º]'], 'outer_T0': 0,
        'planets': [{'mass_earth': p['pl_bmasse'], 'radius_earth': p['pl_rade'],
                      'a': p['pl_orbsmax'], 'e': p['pl_orbeccen']}
                     for _, p in pl_subset.iterrows()],
        'tf': 400, 'dt': dt, 'output_every_n': 10,
        'output_file': f'output/sim_{row["Sustav"].replace(" ", "_")}.hdf5',
    })


In [49]:
result = run_system(configs[0])
print(f"Done: {result}")

OSError: Unable to synchronously create file (unable to truncate a file which is already open)

In [47]:
# quick patch in Colab without re-cloning
!sed -i '/# if we have leftover files/i\        if self.output_name and os.path.isfile(self.output_name):\n            os.remove(self.output_name)' /content/nbody_exoplanets/integrator/data_io.py